In [0]:
# Reading Data and turning them into dataframes 
business_data = spark.read.csv("/Volumes/workspace/default/projectvolume/Business_Data.csv", header=True, inferSchema=True)
store_data =spark.read.csv("/Volumes/workspace/default/projectvolume/Store_Details.csv", header=True, inferSchema=True)

display(business_data)
display(store_data)


Store,Date,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment_Rate,Holiday
1,2017-04-30,42.31,2.572,NA,NA,NA,NA,NA,211.0963582,8.106,false
1,2017-11-30,38.51,2.548,NA,NA,NA,NA,NA,211.2421698,8.106,true
1,2017-02-17,39.93,2.514,NA,NA,NA,NA,NA,211.2891429,8.106,false
1,2017-02-24,46.63,2.561,NA,NA,NA,NA,NA,211.3196429,8.106,false
1,2017-05-01,46.5,2.625,NA,NA,NA,NA,NA,211.3501429,8.106,false
1,2017-12-01,57.79,2.667,NA,NA,NA,NA,NA,211.3806429,8.106,false
1,2017-03-17,54.58,2.72,NA,NA,NA,NA,NA,211.215635,8.106,false
1,2017-03-24,51.45,2.732,NA,NA,NA,NA,NA,211.0180424,8.106,false
1,2017-02-02,62.27,2.719,NA,NA,NA,NA,NA,210.8204499,7.808,false
1,2017-09-02,65.86,2.77,NA,NA,NA,NA,NA,210.6228574,7.808,false


Store,Type,Address,Area_Code,Location,Size
1,E-Commerce Fulfillment,"1893 W Malvern Ave, Fullerton, California",92835,Applegate Ranch Shopping Cente,151315
2,E-Commerce Fulfillment,"1000 Commerce Ave, Atwater, California",95301,Bayfair Cente,202307
3,Food,"15555 East 14th Street, San Leandro, California",94578,Capitola Mal,37392
4,E-Commerce Fulfillment,"1855 41st Avenue, Capitola, California",95010,Chino Spectrum Marketplace�& Towne Cente,205863
5,Food,"3800-4046 Grand Ave.& 3801-4097 Grand Ave., Chino, California",91710,Eagle Rock Plaz,34875
6,E-Commerce Fulfillment,"2700 Colorado Boulevard, Los Angeles, California",90041,Eastvale Gatewa,202505
7,Food,"NWC I-15 and Limonite Ave, Eastvale, California",91752,Escondido Promenad,70713
8,E-Commerce Fulfillment,"1200 Auto Park Way, Escondido, California",92029,Falcon Ridge Town Cente,155078
9,Food,"15320 Summit Ave, Fontana, California",92336,Fallbrook Cente,125833
10,Food,"6633 Fallbrook Ave, West Hills, California",91307,Fremont Hub Shopping Cente,126512


In [0]:
%pip install azure-storage-blob

Looking in indexes: [REDACTED]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# Connect to Azure Blob Storage
from azure.storage.blob import BlobServiceClient

# Azure connection details (using your existing account)
account_url = 'https://jdobbinsutaustin.blob.core.windows.net/'
sas_token = 'sv=2026-02-06&ss=bfqt&srt=sco&sp=rwdlacupiytfx&se=2027-07-26T05:29:22Z&st=2026-07-09T21:14:22Z&spr=https&sig=CejF8LhTUgBFCk5ARIqAH26CVuB3bu7sDnr1sgLnyeU%3D'

# Create blob service client
blob_service_client = BlobServiceClient(account_url=account_url, credential=sas_token)

# Create new container for project data
container_name = 'project-raw'
try:
    container_client = blob_service_client.create_container(container_name)
    print(f"Container '{container_name}' created successfully")
except Exception as e:
    if "ContainerAlreadyExists" in str(e):
        print(f"Container '{container_name}' already exists")
        container_client = blob_service_client.get_container_client(container_name)
    else:
        raise e

Container 'project-raw' already exists


In [0]:
business_data.toPandas().to_csv('/local_disk0/tmp/raw_business_data.csv', index=False)
with open("/local_disk0/tmp/raw_business_data.csv", "rb") as data:
    container_client.upload_blob('raw_business_data.csv', data, overwrite=True)
print("Business data uploaded successfully")

Business data uploaded successfully


In [0]:
store_data.toPandas().to_csv('/local_disk0/tmp/raw_store_details.csv', index=False)
with open("/local_disk0/tmp/raw_store_details.csv", "rb") as data:
    container_client.upload_blob('raw_store_details.csv', data, overwrite=True)
print("Store details uploaded successfully")

Store details uploaded successfully


In [0]:
print('Files in project-raw Container:')
print('-' * 40)
for blob in container_client.list_blobs():
    print(f"Name: {blob['name']}")
    print(f"Size: {blob['size']} bytes")
    print(f"Last Modified: {blob['last_modified']}")
    print('-' * 40)

Files in project-raw Container:
----------------------------------------
Name: raw_business_data.csv
Size: 592506 bytes
Last Modified: 2026-08-16 13:08:49+00:00
----------------------------------------
Name: raw_store_details.csv
Size: 4464 bytes
Last Modified: 2026-08-16 13:08:50+00:00
----------------------------------------
